# bytes と str の違いを知っておく

Python3 では、文字列データを表す方法は、以下2種類

- bytes : バイナリデータ（8ビット値の列）
- str : テキストデータ（Unicode 文字列）

bytes のインスタンスは、生の符号なし 8 ビットの値からなり、通常は ASCIIエンコーディングで表示されます。

In [5]:
a = b'h\x65llo'
print(list(a))
print(a)

[104, 101, 108, 108, 111]
b'hello'


str のインスタンスはテキスト文字を表す Unicode コードポイントを含みます。

In [6]:
a = 'a\u0300 propos' 
print(list(a))
print(a)

['a', '̀', ' ', 'p', 'r', 'o', 'p', 'o', 's']
à propos


重要なことは、str インスタンスはバイナリエンコーディングを持たず、bytes インスタンスはテキストエンコーディングを持たないということです。

つまり、Python ではテキストとバイナリを明確に分けて考える必要がある

### 「str インスタンスはバイナリエンコーディングを持たない」

str は 「文字」そのもの を表しており、「UTF-8」や「Shift_JIS」などのエンコーディング（符号化方式）情報を持たないという意味です。

つまり：
- "こんにちは" は「日本語の文字」列であり、
- それをどんなバイト列（どんなエンコーディング）で表現するかはまだ決まっていません。

In [ ]:
text = "こんにちは"  # str
# ↑ Unicode文字として保持されている

text 自体は、「エンコード方式を知らない」。
→ だから「どのエンコードで保存するか」を指定しないと、ファイル出力できません。

### 「bytes インスタンスはテキストエンコーディングを持たない」

逆に bytes は 「バイト列」 であり、それが「UTF-8の文字列」なのか「JPEG画像」なのかは、Pythonは知らない という意味です。

In [9]:
data = b'\xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf'  # bytes

これはバイト値の列にすぎず、Pythonはこれを「日本語の文字列」とは認識していません。

### だから encode / decode が必要

この2つの世界をつなぐために、明示的に変換する必要があります。

In [10]:
# str → bytes（エンコード）
b = "こんにちは".encode("utf-8")

# bytes → str（デコード）
s = b.decode("utf-8")

> Python プログラムを書くとき、インターフェースの一番遠い箇所で Unicode の符号化と復号化をしておくことが重要です。この方式は、よく Unicode サンドイッチと呼ばれています。プログラムの核心部分では、Unicode データの str 型を使い、文字符号化については一切仮定してはなりません。

### Unicode サンドイッチとは

プログラムの外側（I/O部分）でだけエンコード・デコードを行い、内側（ロジック部分）では常に Unicode（= str）を使う設計

を指します。
つまり、バイナリ→Unicode→バイナリ という構造です。

```pgsql
     +------------------------------+
     |        外部世界（I/O）       |
     |  ファイル / ネットワーク / DB |
     +------------------------------+
                ↓ decode()
        （バイト列 → Unicode文字列）

       🧠 プログラムの内部ロジック
       ---------------------------------
       | ここではすべて str（Unicode）|
       ---------------------------------
                ↑ encode()
        （Unicode文字列 → バイト列）

     +------------------------------+
     |        外部世界（I/O）       |
     +------------------------------+
```

ネットワーク通信やファイルI/Oでは bytes が基本なので、境界（I/O部分）では、bytes に変換

### なぜ「一番外で」やるのが重要なのか

- 内部処理がシンプルになる
  - プログラムのあちこちで .encode() や .decode() を繰り返すと、バグ（TypeError や 文字化け）が発生しやすくなります。
  - Unicodeで統一しておけば、内部では「純粋に文字として」扱えるので、ロジックがシンプルになります。
- エンコーディングの違いに強くなる
  - 外部とのやり取り（ファイルやAPIなど）では、UTF-8 だけでなく Shift_JIS や EUC-JP のような文字コードが混在することがあります。
  - しかし、内部は常に Unicode で統一しておけば、どんな入力でも「decode時に変換して終わり」。内部の処理は全く変えなくて済みます。
- 出力時のエラーを防げる
  - 出力（ファイル保存や通信）では、必ず .encode() が必要です。
  - 内部で常に str を使っていれば、出力時にエンコードを明示的に指定できるため、文字化けや UnicodeEncodeError を防げます。

In [ ]:
def read_file(path):
    # 🍞 外側で decode
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()  # str（Unicode）

def process_text(text):
    # 🧠 内部では常に str（Unicode）
    return text.upper()

def write_file(path, text):
    # 🍞 外側で encode
    with open(path, 'w', encoding='utf-8') as f:
        f.write(text)

# サンドイッチ構造
text = read_file('input.txt')
result = process_text(text)
write_file('output.txt', result)

→ 内部の process_text は文字コードをまったく気にせず書ける。

→ I/O部のみでエンコード／デコードを意識すればよい。

> ここでの教訓は、（Python3 では、'import locale; print(locale.getpreferredencoding())' を使い）システムのデフォルト符号化をチェックして、期待した結果がどうなっているかを理解すべきだということです。心配な場合には、open に対して明示的に encoding パラメータを指定すべきです。

つまり：

- Pythonはファイルを開くときなどに**「システムのデフォルトエンコーディング」**を使うことがある。
- しかし、そのデフォルトは環境によって異なるため、思わぬ文字化けが起きることがある。
- だから「自分の環境では何が使われているかを確認しよう」、
- そして「不安なら明示的に encoding='utf-8' を指定しよう」という教訓です。

### 実務でおすすめの習慣

In [ ]:
# 安全なファイル操作のテンプレート
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# 処理（内部はUnicode）
result = text.upper()

with open("output.txt", "w", encoding="utf-8") as f:
    f.write(result)

## 覚えておくこと

- bytes は 8ビット値の列を含み、str は Unicode コードポイントの文字列を含む
- ヘルパー関数を使って、操作する入力が（8ビット値、UTF-8符号化文字、Unicode コードポイントなど）期待している文字列型になっているかを確かめる
- bytes と str は、> == + % などの演算子では一緒に使えない
- ファイルにバイナリデータを読み書きするには、常に、（'rb' または 'wb' のような）バイナリモードでオープンする
- ファイルに Unicode データを読み書きするには、システムのデフォルトのテキスト符号化に注意する必要がある。問題が生じないように、open では encoding パラメータを明示的に指定する